<a href="https://colab.research.google.com/github/shanusushmita/CS4973-Applied-Multilingual-Systems/blob/main/Python_Example_for_Prompt_Compression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json
import time
import random
import os

# IMPORTANT: The API key is automatically provided in the Canvas environment.
# Do not modify this line.
API_KEY = ""
MODEL_ID = "gemini-2.5-flash-preview-05-20"
API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL_ID}:generateContent?key={API_KEY}"

def generate_responses(prompts):
    """
    Sends a list of prompts to the Gemini API and returns a list of dictionaries
    containing the original prompt and the generated response.
    Includes exponential backoff for robust API calls.
    """
    results = []

    # Process each prompt in the list
    for prompt in prompts:
        retries = 5
        delay = 1
        for i in range(retries):
            try:
                chat_history = []
                chat_history.append({"role": "user", "parts": [{"text": prompt}]})
                payload = {"contents": chat_history}

                print(f"Sending prompt to API: '{prompt[:50]}...'")
                response = requests.post(API_URL, json=payload)
                response.raise_for_status()

                result = response.json()
                if result.get("candidates") and len(result["candidates"]) > 0 and \
                   result["candidates"][0].get("content") and \
                   result["candidates"][0]["content"].get("parts") and \
                   len(result["candidates"][0]["content"]["parts"]) > 0:
                    text = result["candidates"][0]["content"]["parts"][0]["text"]
                    results.append({"prompt": prompt, "response": text})
                    print("...Success.")
                    break  # Exit the retry loop on success
                else:
                    print(f"Warning: Unexpected API response structure on attempt {i+1}.")
                    raise ValueError("Unexpected API response structure.")

            except requests.exceptions.RequestException as e:
                print(f"API call failed on attempt {i+1}: {e}")
                if i < retries - 1:
                    sleep_time = delay * (2 ** i) + random.uniform(0, 1)
                    print(f"Retrying in {sleep_time:.2f} seconds...")
                    time.sleep(sleep_time)
                else:
                    print("All retries failed for this prompt.")
                    results.append({"prompt": prompt, "response": "API ERROR: Failed to get response after multiple retries."})
                    break  # Move to the next prompt
            except ValueError as e:
                print(f"Error parsing API response on attempt {i+1}: {e}")
                results.append({"prompt": prompt, "response": "API ERROR: Failed to parse response."})
                break # Move to the next prompt
    return results

def save_to_jsonl(data, filename="generated_data.jsonl"):
    """
    Saves a list of dictionaries to a JSONL file, with each object on a new line.
    JSONL is an efficient format for storing large datasets.
    """
    print(f"\nSaving generated data to {filename}...")
    try:
        with open(filename, 'w') as f:
            for item in data:
                f.write(json.dumps(item) + '\n')
        print(f"Successfully saved {len(data)} items to {filename}.")
    except IOError as e:
        print(f"Error: Could not write to file '{filename}'. Details: {e}")

def main():
    """
    Demonstrates a scaled data generation workflow using prompt compression
    and efficient JSONL storage.
    """
    # A base article to be summarized
    sample_article = (
        "The history of artificial intelligence (AI) can be traced back to ancient myths about "
        "mechanical men and artificial beings. The field as we know it today was officially "
        "founded at a workshop held on the campus of Dartmouth College in 1956. Over the "
        "following decades, AI research has gone through several cycles of optimism, followed "
        "by funding cuts and reduced interest, known as 'AI winters.' The recent surge in AI "
        "is fueled by advancements in deep learning, increased computational power, and the "
        "availability of massive datasets."
    )

    # Create a list of compressed prompts to simulate a large-scale job
    prompts_to_generate = [
        f"Summarize the following article about the history of AI in 3-4 sentences. Article: {sample_article}",
        f"Summarize the history of AI in one concise paragraph. Article: {sample_article}",
        f"Extract key dates and milestones from this AI history article in a bulleted list. Article: {sample_article}"
    ]

    print("--- Starting a scaled data generation job ---")
    start_time = time.time()

    # Generate responses for all prompts at once
    generated_data = generate_responses(prompts_to_generate)

    end_time = time.time()
    time_taken = end_time - start_time
    print(f"\nTotal time for generation: {time_taken:.2f} seconds")

    # Store the results in a JSONL file
    save_to_jsonl(generated_data, "ai_summary_data.jsonl")

if __name__ == "__main__":
    main()